# LSTM Text Generation on the Epicurious Recipes Dataset

Adapted from Foster, D. (2023). *Generative Deep Learning* (2nd ed.), notebook `05_autoregressive/01_lstm/lstm.ipynb` (https://github.com/davidADSP/Generative_Deep_Learning_2nd_Edition).

**Changes from the textbook notebook (for current Google Colab):**
* Downloads the Epicurious dataset from Kaggle with `kagglehub` instead of reading the book's Docker path `/app/data/...`
* Keras 3 file names: checkpoints end in `.weights.h5` and the saved model ends in `.keras`
* Sampling uses float64 so very low or high temperatures do not break `np.random.choice`
* Final generation uses two temperatures, **0.5** and **1.5** (the assignment excludes 1.0 and 0.2)

**Before running:** Runtime → Change runtime type → **T4 GPU**. Training takes about 15–25 minutes on a GPU.

In [ ]:
# 1. Imports
import numpy as np
import json, re, string, os, glob

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, losses

print('TensorFlow', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
# 2. Parameters (same as the textbook)
VOCAB_SIZE = 10000
MAX_LEN = 200
EMBEDDING_DIM = 100
N_UNITS = 128
VALIDATION_SPLIT = 0.2
SEED = 42
LOAD_MODEL = False
BATCH_SIZE = 32
EPOCHS = 25

TEMPERATURES = [0.5, 1.5]   # the two temperatures for the submission (not 1.0 or 0.2)

np.random.seed(SEED)
tf.random.set_seed(SEED)

## 1. Load the data
The Epicurious dataset is on Kaggle: https://www.kaggle.com/datasets/hugodarwood/epirecipes. The next cell downloads it with `kagglehub`. If the download asks for credentials or fails, download `full_format_recipes.json` from the Kaggle page yourself and upload it with the cell after.

In [ ]:
# Download the Epicurious Recipes dataset from Kaggle
DATA_FILE = os.environ.get('RECIPES_JSON')   # optional override
if not DATA_FILE:
    try:
        import kagglehub
        path = kagglehub.dataset_download('hugodarwood/epirecipes')
        DATA_FILE = glob.glob(os.path.join(path, '**', 'full_format_recipes.json'), recursive=True)[0]
    except Exception as e:
        print('Kaggle download failed:', e)
        print('Run the next cell to upload full_format_recipes.json manually.')
print('Data file:', DATA_FILE)

In [ ]:
# Fallback only: upload full_format_recipes.json yourself (skip if the cell above worked)
if not DATA_FILE or not os.path.exists(DATA_FILE):
    from google.colab import files
    files.upload()
    DATA_FILE = 'full_format_recipes.json'
print('Data file:', DATA_FILE)

In [ ]:
# Load the full dataset
with open(DATA_FILE) as json_data:
    recipe_data = json.load(json_data)

# Filter the dataset: keep recipes with a title and directions
filtered_data = [
    'Recipe for ' + x['title'] + ' | ' + ' '.join(x['directions'])
    for x in recipe_data
    if 'title' in x
    and x['title'] is not None
    and 'directions' in x
    and x['directions'] is not None
]

n_recipes = len(filtered_data)
print(f'{n_recipes} recipes loaded')
print(filtered_data[9])

## 2. Clean up and tokenize the text

In [ ]:
# Pad the punctuation, to treat punctuation marks as separate 'words'
def pad_punctuation(s):
    s = re.sub(f'([{string.punctuation}])', r' \1 ', s)
    s = re.sub(' +', ' ', s)
    return s

text_data = [pad_punctuation(x) for x in filtered_data]
print(text_data[9])

In [ ]:
# Convert to a TensorFlow Dataset
text_ds = (
    tf.data.Dataset.from_tensor_slices(text_data)
    .batch(BATCH_SIZE)
    .shuffle(1000)
)

# Create a vectorization (tokenization) layer: lowercase, top 10,000 words, length 201
vectorize_layer = layers.TextVectorization(
    standardize='lower',
    max_tokens=VOCAB_SIZE,
    output_mode='int',
    output_sequence_length=MAX_LEN + 1,
)

# Adapt the layer to the training set (builds the vocabulary)
vectorize_layer.adapt(text_ds)
vocab = vectorize_layer.get_vocabulary()
VOCAB_SIZE = len(vocab)   # 10,000 on the full dataset; smaller only if the text has fewer distinct words
print('Vocabulary size:', VOCAB_SIZE)

for i, word in enumerate(vocab[:10]):
    print(f'{i}: {word}')

print(vectorize_layer(text_data[9]).numpy())

## 3. Create the training set
Each input is a recipe, and each target is the same recipe shifted one word to the right, so the network learns to predict the next word.

In [ ]:
def prepare_inputs(text):
    text = tf.expand_dims(text, -1)
    tokenized_sentences = vectorize_layer(text)
    x = tokenized_sentences[:, :-1]
    y = tokenized_sentences[:, 1:]
    return x, y

train_ds = text_ds.map(prepare_inputs)

## 4. Build the LSTM

In [ ]:
inputs = layers.Input(shape=(None,), dtype='int32')
x = layers.Embedding(VOCAB_SIZE, EMBEDDING_DIM)(inputs)
x = layers.LSTM(N_UNITS, return_sequences=True)(x)
outputs = layers.Dense(VOCAB_SIZE, activation='softmax')(x)
lstm = models.Model(inputs, outputs)
lstm.summary()

if LOAD_MODEL:
    lstm = models.load_model('./models/lstm.keras', compile=False)

## 5. Train the LSTM

In [ ]:
loss_fn = losses.SparseCategoricalCrossentropy()
lstm.compile('adam', loss_fn)


class TextGenerator(callbacks.Callback):
    """Generates text by repeatedly predicting the next word and appending it."""

    def __init__(self, index_to_word, top_k=10):
        super().__init__()
        self.index_to_word = index_to_word
        self.word_to_index = {word: index for index, word in enumerate(index_to_word)}

    def sample_from(self, probs, temperature):
        # Temperature rescales the distribution: < 1 sharpens it, > 1 flattens it.
        probs = np.asarray(probs, dtype='float64') ** (1 / temperature)
        probs = probs / np.sum(probs)
        return np.random.choice(len(probs), p=probs), probs

    def generate(self, start_prompt, max_tokens, temperature, verbose=True):
        start_tokens = [self.word_to_index.get(x, 1) for x in start_prompt.split()]
        sample_token = None
        info = []
        # 1) feed the existing sequence, 2) predict the next word, 3) append it, repeat
        while len(start_tokens) < max_tokens and sample_token != 0:
            x = np.array([start_tokens])
            y = self.model.predict(x, verbose=0)
            sample_token, probs = self.sample_from(y[0][-1], temperature)
            info.append({'prompt': start_prompt, 'word_probs': probs})
            start_tokens.append(sample_token)
            start_prompt = start_prompt + ' ' + self.index_to_word[sample_token]
        if verbose:
            print(f'\ngenerated text:\n{start_prompt}\n')
        return info, start_prompt

    def on_epoch_end(self, epoch, logs=None):
        self.generate('recipe for', max_tokens=100, temperature=1.0)


os.makedirs('./checkpoint', exist_ok=True)
os.makedirs('./models', exist_ok=True)
model_checkpoint_callback = callbacks.ModelCheckpoint(
    filepath='./checkpoint/checkpoint.weights.h5',   # Keras 3 requires .weights.h5
    save_weights_only=True,
    save_freq='epoch',
    verbose=0,
)
text_generator = TextGenerator(vocab)
train_callbacks = [model_checkpoint_callback, text_generator]
try:
    import tensorboard  # preinstalled in Colab
    train_callbacks.append(callbacks.TensorBoard(log_dir='./logs'))
except ImportError:
    print('TensorBoard not installed: skipping the TensorBoard callback')

In [ ]:
history = lstm.fit(
    train_ds,
    epochs=EPOCHS,
    callbacks=train_callbacks,
)

lstm.save('./models/lstm.keras')   # Keras 3 requires .keras

In [ ]:
# Training loss per epoch
import matplotlib.pyplot as plt
plt.figure(figsize=(7, 4))
plt.plot(range(1, len(history.history['loss']) + 1), history.history['loss'], marker='o')
plt.xlabel('Epoch'); plt.ylabel('Training loss'); plt.title('LSTM training loss')
plt.grid(alpha=0.3); plt.show()

## 6. Generate text at two temperatures
The generator feeds the network an existing sequence, asks it to predict the next word, appends that word, and repeats. The same prompts are run at both temperatures so the outputs can be compared.

In [ ]:
def print_probs(info, vocab, top_k=5):
    for i in info:
        print(f"\nPROMPT: {i['prompt']}")
        word_probs = i['word_probs']
        p_sorted = np.sort(word_probs)[::-1][:top_k]
        i_sorted = np.argsort(word_probs)[::-1][:top_k]
        for p, idx in zip(p_sorted, i_sorted):
            print(f'{vocab[idx]}:   \t{np.round(100 * p, 2)}%')
        print('--------')


PROMPTS = [
    'recipe for roasted vegetables |',
    'recipe for chocolate ice cream |',
    'recipe for grilled chicken with lemon |',
]

for temperature in TEMPERATURES:
    print('=' * 90)
    print(f'TEMPERATURE = {temperature}')
    print('=' * 90)
    for prompt in PROMPTS:
        text_generator.generate(prompt, max_tokens=100, temperature=temperature)

### Next-word probabilities at each temperature
This shows *why* the outputs differ: a lower temperature concentrates probability on the most likely words, and a higher temperature spreads it out.

In [ ]:
for temperature in TEMPERATURES:
    print('=' * 90)
    print(f'TEMPERATURE = {temperature}')
    print('=' * 90)
    info, _ = text_generator.generate('recipe for roasted vegetables | chop 1 /', max_tokens=10, temperature=temperature)
    print_probs(info, vocab)